# WayTooCooked — RL Training Notebook

Trains a DQN agent (via `stable_baselines3`) to make and deliver a burger in an
Overcooked-like kitchen environment.

## Fixes applied over the raw `mathou_env` branch
| Bug | Fix |
|-----|-----|
| Orders start empty → delivery always fails | `reset()` now starts with one active order |
| Broken ingredient-count anti-hack (never decreases) | Replaced with per-episode milestone flags |
| `held_obj` not cleared after placing steak on dish | Added `self.held_obj = None` + pan returned to surface |
| Debug `print()` flooding training output | Removed |
| Timeout returned `done=True` (Gymnasium violation) | Now returns `truncated=True` |
| Chopping completion gave no reward | Added `+1.0` milestone reward |

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback


## Environment

Key design decisions carried over from `mathou_env`:
- **Normalized, one-hot observation space** — all values in `[0, 1]`
- **Pan-carrying mechanic** — agent picks up the whole pan when steak is ready
- **Burn mechanic** — steak burns if left in pan too long after cooking
- **Distance shaping** — small reward for moving toward dish while carrying an ingredient
- **Order system** — one burger order starts active; delivery only succeeds with an active order
- **Counter tiles (`.`)** — perimeter counters allow safe item storage; dropping on empty floor loses the item

In [ ]:
class OvercookedKitchen(gym.Env):
    def __init__(self, layout):
        self.layout = layout
        self.height = len(layout)
        self.width = len(layout[0])
        self.cooking_surfaces = [
            (x, y) for y, row in enumerate(layout)
            for x, tile in enumerate(row) if tile == 'P'
        ]
        count = 1
        self.chopping_boards = {}
        for y, row in enumerate(layout):
            for x, tile in enumerate(row):
                if tile == 'C':
                    self.chopping_boards[(x, y)] = f'chop_board_{count}'
                    count += 1
        self.dish_positions = [
            (x, y) for y, row in enumerate(layout)
            for x, tile in enumerate(row) if tile == 'D'
        ]
        self.dispensers = {
            'tomato': [(x, y) for y, row in enumerate(layout) for x, tile in enumerate(row) if tile == 'T'],
            'steak':  [(x, y) for y, row in enumerate(layout) for x, tile in enumerate(row) if tile == 'S'],
            'bread':  [(x, y) for y, row in enumerate(layout) for x, tile in enumerate(row) if tile == 'B'],
        }
        self.TIME_TO_CHOP = 3
        self.TIME_TO_COOK = 5
        self.TIME_TO_BURN = 5
        self.MAX_ORDERS = 1
        self.MAX_STEPS = 200
        self.MAX_ORDER_TIME = 100
        # Actions: 0-3 Movement, 4 Interact, 5 Chop
        self.action_space = spaces.Discrete(6)
        # Observation: agent_pos(2) + dir_onehot(4) + held_onehot(8)
        #              + per_pan(pos(2)+status_onehot(3)+timer(1))
        #              + per_dish(pos(2)+ingredients(3))
        #              + per_board(pos(2)+progress(1)+obj_onehot(3))
        #              + ingredient_distances(3) + pan_dist(1) + dish_dist(1) + time(1)
        obs_size = (
            2 + 4 + 8
            + len(self.cooking_surfaces) * 6
            + len(self.dish_positions) * 5
            + len(self.chopping_boards) * 6
            + 3 + 1 + 1 + 1
        )
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(obs_size,), dtype=np.float32)
        print(f"Observation space size: {obs_size}")

    # ------------------------------------------------------------------
    def _milestone_reward(self, name, value):
        """Return `value` the first time `name` milestone is hit, 0 thereafter."""
        if name not in self.milestones:
            self.milestones.add(name)
            return value
        return 0.0

    # ------------------------------------------------------------------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0
        self.agent_pos = (2, 2)
        self.agent_dir = (0, 1)
        self.held_obj = None
        self.done = False

        # Milestone flags: each sub-task reward is given only once per episode
        self.milestones = set()

        self.objects_on_counters = {}
        self.pans_state = {}
        self.dishes = {}
        self.chopping_boards_state = {}

        # Always start with one active burger order so delivery can always succeed
        self.orders = {'order_0': {'recipe': 'burger', 'timer': 0}}

        for y, row in enumerate(self.layout):
            for x, tile in enumerate(row):
                if x == 0 or x == self.width - 1 or y == 0 or y == self.height - 1:
                    self.objects_on_counters[(x, y)] = None

        for i, pan_pos in enumerate(self.cooking_surfaces):
            self.pans_state[f'pan_{i}'] = {'location': pan_pos, 'status': 'empty', 'timer': 0}
            self.objects_on_counters[pan_pos] = f'pan_{i}'

        for i, dish_pos in enumerate(self.dish_positions):
            self.dishes[f'dish_{i}'] = {'location': dish_pos, 'bread': False, 'tomato': False, 'steak': False}
            self.objects_on_counters[dish_pos] = f'dish_{i}'

        for board_id in self.chopping_boards.values():
            self.chopping_boards_state[board_id] = 0

        return self._get_obs(), {}

    # ------------------------------------------------------------------
    def render(self):
        grid_copy = [list(row) for row in self.layout]
        for (x, y), obj in self.objects_on_counters.items():
            if obj is not None:
                grid_copy[y][x] = obj[0].upper()
        for ingredient, positions in self.dispensers.items():
            char = ingredient[0].upper()
            for (x, y) in positions:
                grid_copy[y][x] = char
        ax, ay = self.agent_pos
        dir_map = {(0, -1): '^', (0, 1): 'v', (-1, 0): '<', (1, 0): '>'}
        grid_copy[ay][ax] = dir_map.get(self.agent_dir, 'A')
        print('\n' + '='*20)
        for row in grid_copy:
            print(' '.join(row))
        print(f'Hand : {self.held_obj}')
        print(f'Pans : {[(p["location"], p["status"], p["timer"]) for p in self.pans_state.values()]}')
        print(f'Dishes : {[(d["location"], d["bread"], d["tomato"], d["steak"]) for d in self.dishes.values()]}')
        print(f'Boards : {[(c, self.chopping_boards_state[c]) for c in self.chopping_boards_state]}')
        active = {oid: o for oid, o in self.orders.items() if o["recipe"] != "empty"}
        print(f'Orders : {len(active)} active | {active}')
        print(f'Steps  : {self.step_count}/{self.MAX_STEPS}')
        print('='*20)

    # ------------------------------------------------------------------
    def _get_obs(self):
        obs = []
        ax, ay = self.agent_pos

        # Agent position (normalized)
        obs.append(ax / self.width)
        obs.append(ay / self.height)

        # Direction one-hot [up, down, left, right]
        dir_vec = [0, 0, 0, 0]
        if self.agent_dir == (0, -1):   dir_vec[0] = 1
        elif self.agent_dir == (0, 1):  dir_vec[1] = 1
        elif self.agent_dir == (-1, 0): dir_vec[2] = 1
        elif self.agent_dir == (1, 0):  dir_vec[3] = 1
        obs.extend(dir_vec)

        # Held object one-hot [none, tomato, steak, bread, chopped_tomato, burger, pan_empty, pan_ready]
        held = [0] * 8
        if self.held_obj is None:                  held[0] = 1
        elif self.held_obj == 'tomato':             held[1] = 1
        elif self.held_obj == 'steak':             held[2] = 1
        elif self.held_obj == 'bread':             held[3] = 1
        elif self.held_obj == 'chopped_tomato':    held[4] = 1
        elif self.held_obj == 'full_burger':       held[5] = 1
        elif 'pan' in str(self.held_obj):
            pan = self.pans_state[self.held_obj]
            if pan['status'] == 'ready':
                held[7] = 1
            else:
                held[6] = 1
        obs.extend(held)

        # Pan states: position + status one-hot + cooking progress
        for pan in self.pans_state.values():
            px, py = pan['location']
            obs.append(px / self.width)
            obs.append(py / self.height)
            if pan['status'] == 'empty':       obs.extend([1, 0, 0])
            elif pan['status'] == 'cooking':   obs.extend([0, 1, 0])
            else:                              obs.extend([0, 0, 1])
            obs.append(min(pan['timer'] / self.TIME_TO_COOK, 1.0))

        # Dish states: position + ingredient flags
        for dish in self.dishes.values():
            dx, dy = dish['location']
            obs.append(dx / self.width)
            obs.append(dy / self.height)
            obs.append(int(dish['bread']))
            obs.append(int(dish['tomato']))
            obs.append(int(dish['steak']))

        # Chopping board states: position + progress + object one-hot
        for pos, board_id in self.chopping_boards.items():
            x, y = pos
            obs.append(x / self.width)
            obs.append(y / self.height)
            obs.append(self.chopping_boards_state[board_id] / self.TIME_TO_CHOP)
            obj = self.objects_on_counters.get(pos)
            board_obj = [0, 0, 0]  # [empty, tomato, chopped_tomato]
            if obj is None:                 board_obj[0] = 1
            elif obj == 'tomato':           board_obj[1] = 1
            elif obj == 'chopped_tomato':   board_obj[2] = 1
            obs.extend(board_obj)

        # Distances to ingredient dispensers (normalized Manhattan)
        for ingredient in ['tomato', 'steak', 'bread']:
            dist = min(abs(ax - x) + abs(ay - y) for x, y in self.dispensers[ingredient])
            obs.append(dist / (self.width + self.height))

        # Distance to nearest pan
        pan_dist = min(abs(ax - p['location'][0]) + abs(ay - p['location'][1]) for p in self.pans_state.values())
        obs.append(pan_dist / (self.width + self.height))

        # Distance to nearest dish
        dish_dist = min(abs(ax - d['location'][0]) + abs(ay - d['location'][1]) for d in self.dishes.values())
        obs.append(dish_dist / (self.width + self.height))

        # Normalised time
        obs.append(self.step_count / self.MAX_STEPS)

        return np.array(obs, dtype=np.float32)

    # ------------------------------------------------------------------
    def _get_adj_pos(self):
        x, y = self.agent_pos
        dx, dy = self.agent_dir
        return (x + dx, y + dy)

    # ------------------------------------------------------------------
    def step(self, action):
        reward = -0.01  # Per-step survival penalty

        # Passive cooking
        for _, pan in self.pans_state.items():
            if pan['status'] == 'cooking' and pan['location'] in self.cooking_surfaces:
                pan['timer'] += 1
                if pan['timer'] == self.TIME_TO_COOK:
                    pan['status'] = 'ready'
                    reward += self._milestone_reward('steak_cooked', 1.0)
                elif pan['timer'] >= self.TIME_TO_COOK + self.TIME_TO_BURN:
                    pan['status'] = 'empty'
                    pan['timer'] = 0
                    reward -= 0.5  # Penalty for burning the steak

        # Order timers
        for order in self.orders.values():
            if order['recipe'] != 'empty':
                order['timer'] += 1

        if action < 4:  # MOVEMENT
            reward += self._handle_move(action)

        elif action == 4:  # INTERACT
            target_pos = self._get_adj_pos()
            tile = self.layout[target_pos[1]][target_pos[0]]

            if target_pos in self.dispensers['tomato'] and self.held_obj is None:
                self.held_obj = 'tomato'
                reward += self._milestone_reward('picked_tomato', 0.3)
            elif target_pos in self.dispensers['steak'] and self.held_obj is None:
                self.held_obj = 'steak'
                reward += self._milestone_reward('picked_steak', 0.3)
            elif target_pos in self.dispensers['bread'] and self.held_obj is None:
                self.held_obj = 'bread'
                reward += self._milestone_reward('picked_bread', 0.3)
            elif tile == 'P':
                reward += self._handle_pan_interact(target_pos)
            elif tile == ' ':  # Empty floor: drop item (item is lost)
                reward += self._handle_empty_space_interact()
            elif tile == 'D':
                reward += self._handle_dish_interact(target_pos)
            elif tile == 'C':
                reward += self._handle_chopping_board_interact(target_pos)
            elif tile == 'V':
                reward += self._handle_delivery()
            elif tile == '.':
                reward += self._handle_counter_interact(target_pos)

        elif action == 5 and self.held_obj is None:  # CHOP
            target_pos = self._get_adj_pos()
            if self.layout[target_pos[1]][target_pos[0]] == 'C':
                reward += self._handle_chopping(target_pos)

        self.step_count += 1
        # Proper Gymnasium semantics: timeout → truncated, delivery → terminated
        truncated = (self.step_count >= self.MAX_STEPS) and not self.done
        return self._get_obs(), reward, self.done, truncated, {}

    # ------------------------------------------------------------------
    def _handle_move(self, action):
        reward = 0.0
        directions = [(0, -1), (0, 1), (-1, 0), (1, 0)]
        dx, dy = directions[action]
        self.agent_dir = (dx, dy)
        old_x, old_y = self.agent_pos
        new_x, new_y = old_x + dx, old_y + dy
        if 0 <= new_x < self.width and 0 <= new_y < self.height:
            if self.layout[new_y][new_x] == ' ':
                self.agent_pos = (new_x, new_y)
                if 'pan' in str(self.held_obj):
                    self.pans_state[self.held_obj]['location'] = (new_x, new_y)
        # Distance shaping: reward getting closer to dish while carrying ingredient/pan
        if self.held_obj in ('chopped_tomato', 'bread') or 'pan' in str(self.held_obj):
            new_dist  = min(abs(self.agent_pos[0] - d[0]) + abs(self.agent_pos[1] - d[1]) for d in self.dish_positions)
            old_dist  = min(abs(old_x - d[0]) + abs(old_y - d[1]) for d in self.dish_positions)
            if new_dist < old_dist:
                reward += 0.01
        return reward

    # ------------------------------------------------------------------
    def _handle_chopping_board_interact(self, target_pos):
        obj_on_board = self.objects_on_counters.get(target_pos)
        reward = 0.0
        if obj_on_board is None and self.held_obj == 'tomato':
            self.objects_on_counters[target_pos] = 'tomato'
            self.held_obj = None
            self.chopping_boards_state[self.chopping_boards[target_pos]] = 0
            reward += self._milestone_reward('placed_tomato_on_board', 0.3)
        elif obj_on_board == 'chopped_tomato' and self.held_obj is None:
            self.held_obj = 'chopped_tomato'
            self.objects_on_counters[target_pos] = None
            self.chopping_boards_state[self.chopping_boards[target_pos]] = 0
            reward += self._milestone_reward('picked_chopped_tomato', 0.3)
        return reward

    # ------------------------------------------------------------------
    def _handle_empty_space_interact(self):
        """Dropping an item on the empty floor loses it (intentional design)."""
        reward = 0.0
        if self.held_obj is not None:
            self.held_obj = None
            reward -= 0.5
        return reward

    # ------------------------------------------------------------------
    def _handle_counter_interact(self, target_pos):
        """Place or pick up items on perimeter counter tiles."""
        reward = 0.0
        obj_on_tile = self.objects_on_counters.get(target_pos)
        if self.held_obj is not None:
            if obj_on_tile is None:
                if 'pan' in str(self.held_obj):
                    self.pans_state[self.held_obj]['location'] = target_pos
                    if target_pos in self.cooking_surfaces:
                        reward += 0.15
                self.objects_on_counters[target_pos] = self.held_obj
                self.held_obj = None
                reward -= 0.1  # Small penalty for putting item down instead of using it
        elif obj_on_tile is not None:
            self.held_obj = obj_on_tile
            self.objects_on_counters[target_pos] = None
        return reward

    # ------------------------------------------------------------------
    def _handle_pan_interact(self, target_pos):
        reward = 0.0
        pan_id_idx = self.cooking_surfaces.index(target_pos)
        pan_id = f'pan_{pan_id_idx}'
        pan = self.pans_state[pan_id]

        if self.held_obj == 'steak' and pan['status'] == 'empty' and pan['location'] == target_pos:
            pan['status'] = 'cooking'
            pan['timer'] = 0
            self.held_obj = None
            reward += self._milestone_reward('placed_steak_in_pan', 0.5)
        elif pan['status'] == 'ready' and self.held_obj is None:
            self.held_obj = pan_id
            self.objects_on_counters[target_pos] = None
            reward += self._milestone_reward('picked_cooked_steak', 0.5)
        elif self.held_obj == pan_id and self.objects_on_counters.get(target_pos) is None:
            # Put pan back on pan tile
            self.held_obj = None
            self.objects_on_counters[target_pos] = pan_id
            pan['location'] = target_pos
        return reward

    # ------------------------------------------------------------------
    def _handle_dish_interact(self, target_pos):
        reward = 0.0
        pan_loc = self.cooking_surfaces[0]  # Default cooking surface for pan return
        id_dish = self.dish_positions.index(target_pos)
        dish = self.dishes[f'dish_{id_dish}']

        if self.held_obj:
            if self.held_obj == 'bread' and not dish['bread']:
                dish['bread'] = True
                self.held_obj = None
                reward += self._milestone_reward('placed_bread_on_dish', 2.0)

            elif self.held_obj == 'chopped_tomato' and not dish['tomato'] and dish['bread']:
                dish['tomato'] = True
                self.held_obj = None
                reward += self._milestone_reward('placed_tomato_on_dish', 2.0)

            elif 'pan' in str(self.held_obj) and not dish['steak'] and dish['bread']:
                pan_id = self.held_obj
                dish['steak'] = True
                # BUG FIX: clear held_obj and return empty pan to cooking surface
                self.pans_state[pan_id]['status'] = 'empty'
                self.pans_state[pan_id]['timer'] = 0
                self.pans_state[pan_id]['location'] = pan_loc
                self.objects_on_counters[pan_loc] = pan_id
                self.held_obj = None
                reward += self._milestone_reward('placed_steak_on_dish', 2.0)

        elif all(dish.values()) and self.held_obj is None:
            # Pick up the completed burger
            self.held_obj = 'full_burger'
            for key in dish:
                dish[key] = False
            reward += self._milestone_reward('assembled_burger', 5.0)

        return reward

    # ------------------------------------------------------------------
    def _handle_chopping(self, target_pos):
        reward = 0.0
        obj = self.objects_on_counters.get(target_pos)
        board_id = self.chopping_boards.get(target_pos)
        if obj == 'tomato' and board_id is not None:
            self.chopping_boards_state[board_id] += 1
            if self.chopping_boards_state[board_id] >= self.TIME_TO_CHOP:
                self.objects_on_counters[target_pos] = 'chopped_tomato'
                self.chopping_boards_state[board_id] = 0
                reward += self._milestone_reward('chopped_tomato_done', 1.0)
        return reward

    # ------------------------------------------------------------------
    def _handle_delivery(self):
        # Delivery reward (50.0) is much larger than milestone rewards so the
        # agent prioritises completing orders over merely collecting ingredients.
        if self.held_obj == 'full_burger' and any(
            order['recipe'] == 'burger' for order in self.orders.values()
        ):
            self.held_obj = None
            self.done = True
            self.orders = {}
            return 50.0
        return -1.0  # Penalty for wrong/empty delivery attempt


In [ ]:
# Layout Legend:
# X: Wall (impassable, no interaction)
# T: Tomato dispenser, S: Steak dispenser, B: Bread dispenser
# C: Chopping board, P: Pan (cooking surface)
# D: Dish (assembly point), V: Delivery point
# .: Counter (items can be stored here)
#  : Empty floor (agent can walk here; dropping items here loses them)

MY_LAYOUT = [
    "..T.S.B.X",
    "C       P",
    ".       .",
    ".       V",
    "....D...."
]

# Sanity check
env = OvercookedKitchen(MY_LAYOUT)
obs, _ = env.reset()
print(f"Observation space: {env.observation_space}")
print(f"Action space:      {env.action_space}")
print(f"Initial obs shape: {obs.shape}")
assert env.observation_space.contains(obs), "Initial obs out of bounds!"

# Quick smoke test: 500 random steps, all observations must stay in bounds
for _ in range(500):
    a = env.action_space.sample()
    obs, r, done, trunc, _ = env.step(a)
    assert env.observation_space.contains(obs), f"Obs out of bounds: {obs}"
    if done or trunc:
        obs, _ = env.reset()
print("Smoke test passed (500 random steps, all obs in bounds)")


## Training with Stable-Baselines3 DQN

We use a `BaseCallback` to track per-episode rewards and success rate during training.

In [ ]:
class EpisodeLogger(BaseCallback):
    """Records per-episode total rewards and success flags during SB3 training."""

    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.episode_successes = []
        self._current_reward = 0.0

    def _on_step(self) -> bool:
        reward = self.locals['rewards'][0]
        done   = self.locals['dones'][0]
        info   = self.locals['infos'][0]
        self._current_reward += float(reward)
        if done:
            # SB3 passes the real terminal info under 'final_info' when using VecEnv
            success = info.get('is_success', False)
            self.episode_rewards.append(self._current_reward)
            self.episode_successes.append(float(success))
            self._current_reward = 0.0
            if len(self.episode_rewards) % 100 == 0:
                last = self.episode_rewards[-100:]
                succ = self.episode_successes[-100:]
                print(f"  Ep {len(self.episode_rewards):4d} | "
                      f"avg reward: {sum(last)/len(last):7.2f} | "
                      f"success: {sum(succ)/len(succ)*100:5.1f}%")
        return True


In [ ]:
env = OvercookedKitchen(MY_LAYOUT)

logger = EpisodeLogger()

model = DQN(
    'MlpPolicy',
    env,
    verbose=0,
    learning_rate=1e-4,
    buffer_size=100_000,
    learning_starts=1_000,
    batch_size=64,
    gamma=0.99,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.01,
    exploration_fraction=0.6,
    policy_kwargs=dict(net_arch=[256, 256]),
)

print("Training DQN for 200 000 timesteps...")
model.learn(total_timesteps=200_000, callback=logger, progress_bar=False)
model.save('model')
print("\nTraining complete — model saved to model.zip")


## Analysis

### Learning Curve
Smooth the episode reward history with a rolling window.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt

WINDOW = 50
ep_rewards = logger.episode_rewards

if len(ep_rewards) >= WINDOW:
    kernel = np.ones(WINDOW) / WINDOW
    smoothed = np.convolve(ep_rewards, kernel, mode='valid')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(smoothed)
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('Avg reward (smoothed)')
    axes[0].set_title(f'Learning curve (window={WINDOW})')

    succ = np.convolve(logger.episode_successes, kernel, mode='valid') * 100
    axes[1].plot(succ, color='green')
    axes[1].set_xlabel('Episode')
    axes[1].set_ylabel('Success rate %')
    axes[1].set_title('Success rate')

    plt.tight_layout()
    plt.savefig('learning_curve.png', dpi=100)
    plt.show()
    print(f"Final avg reward (last {WINDOW} ep): {np.mean(ep_rewards[-WINDOW:]):.2f}")
    print(f"Best episode reward: {max(ep_rewards):.2f}")
else:
    print(f'Only {len(ep_rewards)} episodes recorded; increase total_timesteps for a learning curve.')


### Action Distribution in Replay Buffer

Shows which actions the agent took most frequently (helps spot degenerate policies).

In [ ]:
buffer = model.replay_buffer
actions = buffer.actions[:buffer.pos]
unique, counts = np.unique(actions, return_counts=True)
action_names = ['Up', 'Down', 'Left', 'Right', 'Interact', 'Chop']
print('Action distribution in replay buffer:')
for u, c in zip(unique, counts):
    bar = '#' * (c * 40 // counts.max())
    print(f'  {action_names[int(u)]:<10} ({int(u)}): {c:6d}  {bar}')


### Replay Best Episode

Find the episode with the highest total reward and replay it step by step
using the actions stored in the replay buffer.

In [ ]:
def get_actions_from_episode(model, episode_idx):
    buf = model.replay_buffer
    actions = buf.actions[:buf.pos]
    dones = buf.dones[:buf.pos]
    episodes, current = [], []
    for i in range(len(actions)):
        current.append(int(actions[i][0][0]))
        if dones[i]:
            episodes.append(current)
            current = []
    if episode_idx >= len(episodes):
        print(f'Buffer contains only {len(episodes)} complete episodes.')
        return None
    return episodes[episode_idx]


def replay_episode(env, actions):
    obs, _ = env.reset()
    env.render()
    total_r = 0.0
    action_names = ['Up', 'Down', 'Left', 'Right', 'Interact', 'Chop']
    for a in actions:
        obs, r, done, trunc, _ = env.step(a)
        total_r += r
        env.render()
        print(f'  Action: {action_names[a]:<10} | step reward: {r:+.3f} | total: {total_r:+.3f}')
        if done or trunc:
            break
    status = 'SUCCESS 🎉' if 'assembled_burger' in env.milestones else 'did not finish'
    print(f'\nEpisode result: {status}')
    print(f'Milestones: {env.milestones}')


if logger.episode_rewards:
    best_ep_idx = int(np.argmax(logger.episode_rewards))
    print(f'Best episode: #{best_ep_idx} with reward {logger.episode_rewards[best_ep_idx]:.2f}')
    best_actions = get_actions_from_episode(model, best_ep_idx)
    if best_actions:
        replay_episode(env, best_actions)


### Greedy Evaluation (ε = 0)

Run a fresh episode with the fully-greedy trained policy.

In [ ]:
eval_env = OvercookedKitchen(MY_LAYOUT)
obs, _ = eval_env.reset()
eval_env.render()

total_eval_reward = 0.0
action_names = ['Up', 'Down', 'Left', 'Right', 'Interact', 'Chop']
for step in range(eval_env.MAX_STEPS):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, _ = eval_env.step(int(action))
    eval_env.render()
    print(f'Step {step+1:3d}: {action_names[int(action)]:<10} | reward: {reward:+.3f}')
    total_eval_reward += reward
    if done or truncated:
        break

status = 'SUCCESS 🎉' if 'assembled_burger' in eval_env.milestones else 'did not finish'
print(f'\nEvaluation result: {status}')
print(f'Total reward: {total_eval_reward:.2f}')
print(f'Milestones: {eval_env.milestones}')
